In [98]:
import json
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

In [99]:
state_file = Path("selected_data.json")
if not state_file.exists():
    raise FileNotFoundError(
        "Run app.py, upload a dataset, and select a target column first."
    )

with state_file.open(encoding="utf-8") as file:
    selected_data = json.load(file)

uploaded_file_name = selected_data["uploaded_file_name"]
data_file = Path("uploads") / uploaded_file_name

if not data_file.exists():
    raise FileNotFoundError(f"Uploaded file not found: {data_file}")

if data_file.suffix.lower() == ".csv":
    df = pd.read_csv(data_file)
elif data_file.suffix.lower() in {".xlsx", ".xls"}:
    df = pd.read_excel(data_file)
else:
    df = pd.read_parquet(data_file)

print(f"Uploaded file: {uploaded_file_name}")
df.head()

Uploaded file: data.csv


,brand,color,size,price,offer_price
0,Tresmode,Black,6.5,4647,4153
1,Lavie,Gold,4,2446,1172
2,FILA,Red,6,2346,1898
3,Crocs,Red,5,5942,5868
4,Crocs,Blue,4,2942,2118


In [100]:
target_col = selected_data["target_col"]

if target_col not in df.columns:
    raise ValueError(
        f"Target column {target_col!r} was not found. "
        f"Available columns: {list(df.columns)}"
    )

# Select the target values.
y = df[target_col]

if y.dtype == "object" or y.nunique() <= 10:
    problem_type = "classification"
else:
    problem_type = "regression"

print(f"Target column: {target_col}")
print(f"Problem type: {problem_type}")

Target column: price
Problem type: regression


In [101]:
df.shape

(5679, 5)

In [102]:
df.head(5)

,brand,color,size,price,offer_price
0,Tresmode,Black,6.5,4647,4153
1,Lavie,Gold,4,2446,1172
2,FILA,Red,6,2346,1898
3,Crocs,Red,5,5942,5868
4,Crocs,Blue,4,2942,2118


In [103]:
# Remove rows containing missing values

df = df.dropna()
# Remove duplicate rows
df = df.drop_duplicates()

numeric_columns = df.select_dtypes(include="number").columns


# Remove rows containing values above the 99th percentile
outlier_mask = (
    df[numeric_columns] <= df[numeric_columns].quantile(0.99)
).all(axis=1)

df = df.loc[outlier_mask].copy()
y = df[target_col]

# Display the cleaned dataframe
print(f"\nCleaned dataframe shape: {df.shape}")
df.head()


Cleaned dataframe shape: (4592, 5)


,brand,color,size,price,offer_price
0,Tresmode,Black,6.5,4647,4153
1,Lavie,Gold,4,2446,1172
2,FILA,Red,6,2346,1898
3,Crocs,Red,5,5942,5868
4,Crocs,Blue,4,2942,2118


In [104]:
import re

# Identify and remove ID/name/email/phone columns
identifier_pattern = re.compile(
    r"(^|[_\s-])(id|name|email|phone|phone_num|phone_number)([_\s-]|$)",
    re.IGNORECASE,
)

columns_to_remove = [
    column for column in df.columns
    if identifier_pattern.search(str(column))
]

df = df.drop(columns=columns_to_remove, errors="ignore")

# One-hot encode remaining object/category columns
categorical_columns = df.select_dtypes(
    include=["object", "category"]
).columns

df = pd.get_dummies(
    df,
    columns=categorical_columns,
    dtype=int
)

# Update target values
y = df[target_col]

print(f"Removed columns: {columns_to_remove}")
print(f"Encoded dataframe shape: {df.shape}")

df.head()

Removed columns: []
Encoded dataframe shape: (4592, 185)


C:\Users\Md Mahfuzur Rahman\AppData\Local\Temp\ipykernel_22240\1977386658.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


,price,offer_price,brand_Adidas,brand_Asics,brand_Bata,brand_Buckaroo,brand_Converse,brand_Crocs,brand_FILA,brand_Footin,...,size_8,size_8.5,size_9,size_9.5,size_UK-06,size_UK-07,size_UK-08,size_UK-09,size_UK-10,size_UK-11
0,4647,4153,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2446,1172,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2346,1898,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,5942,5868,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2942,2118,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [105]:
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=[target_col])
y = df[target_col]

x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y if problem_type == "classification" else None,
)

scaler = StandardScaler()

x_train = pd.DataFrame(
    scaler.fit_transform(x_train),
    columns=x_train.columns,
    index=x_train.index,
)

x_test = pd.DataFrame(
    scaler.transform(x_test),
    columns=x_test.columns,
    index=x_test.index,
)

print(f"x_train: {x_train.shape}")
print(f"x_test: {x_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

x_train: (3673, 184)
x_test: (919, 184)
y_train: (3673,)
y_test: (919,)


In [106]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVR, SVC

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    RandomForestClassifier,
    GradientBoostingClassifier,
)

if problem_type == "regression":
    models = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(
            n_estimators=200, random_state=42
        ),
        "Gradient Boosting": GradientBoostingRegressor(random_state=42),
        "SVR": SVR(),
    }
else:
    models = {
        "Logistic Regression": LogisticRegression(
            max_iter=1000, random_state=42
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=200, random_state=42
        ),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42),
        "SVM": SVC(random_state=42),
    }

result_df = pd.DataFrame(
    {"actual": y_test.to_numpy()},
    index=y_test.index,
)

for model_name, model in models.items():
    model.fit(x_train, y_train)
    result_df[model_name] = model.predict(x_test)

print(f"Trained {len(models)} {problem_type} models.")
result_df.head()

Trained 4 regression models.


,actual,Linear Regression,Random Forest,Gradient Boosting,SVR
3730,3442,3297.559001,3437.105,3213.007580,3716.769030
3230,4442,3604.898847,4442.000,3972.800717,3702.192891
1763,1346,1320.418512,1347.500,1639.236884,3726.204368
3081,2142,2467.671263,2142.000,2113.040569,3706.870261
4947,3747,3623.426634,3679.395,3787.996138,3731.276093


In [107]:
import numpy as np
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, r2_score

model_names = list(models.keys())

if problem_type == "classification":
    metrics_df = pd.DataFrame(
        {
            "model": model_names,
            "accuracy": [
                accuracy_score(result_df["actual"], result_df[model_name])
                for model_name in model_names
            ],
        }
    ).sort_values("accuracy", ascending=False)
else:
    metrics_df = pd.DataFrame(
        {
            "model": model_names,
            "r2": [
                r2_score(result_df["actual"], result_df[model_name])
                for model_name in model_names
            ],
            "mae": [
                mean_absolute_error(result_df["actual"], result_df[model_name])
                for model_name in model_names
            ],
            "rmse": [
                np.sqrt(
                    mean_squared_error(
                        result_df["actual"], result_df[model_name]
                    )
                )
                for model_name in model_names
            ],
        }
    ).sort_values("r2", ascending=False)

print(f"{problem_type.title()} model performance:")
metrics_df

Regression model performance:


,model,r2,mae,rmse
1,Random Forest,0.996549,120.305621,309.937632
2,Gradient Boosting,0.991992,347.967731,472.141418
0,Linear Regression,0.987629,395.587447,586.835030
3,SVR,-0.054335,2184.623844,5417.542192


In [108]:
import joblib

model_parameters_folder = Path("model_parameters")
model_parameters_folder.mkdir(parents=True, exist_ok=True)

best_model_name = metrics_df.iloc[0]["model"]
best_model = models[best_model_name]

model_bundle = {
    "model_name": best_model_name,
    "model": best_model,
    "scaler": scaler,
    "feature_columns": list(x_train.columns),
    "target_column": target_col,
    "problem_type": problem_type,
}

model_path = model_parameters_folder / "best_model.joblib"
joblib.dump(model_bundle, model_path)

print(f"Best model: {best_model_name}")
print(f"Saved to: {model_path}")

Best model: Random Forest
Saved to: model_parameters\best_model.joblib


In [109]:
best_prediction_column = best_model_name

best_model_results = {
    "model_name": best_model_name,
    "problem_type": problem_type,
    "target_column": target_col,
    "metrics": metrics_df.loc[
        metrics_df["model"] == best_model_name
    ].iloc[0].to_dict()
}

results_path = Path("best_model_results.json")
with results_path.open("w", encoding="utf-8") as file:
    json.dump(best_model_results, file, indent=4, default=str)

print(f"Best model results saved to: {results_path}")

Best model results saved to: best_model_results.json
